In [ ]:
#%%
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
#load the dataset
from folktexts.acs.acs_tasks import ACSTaskMetadata
from folktexts.acs.acs_dataset import ACSDataset
from folktexts.baseline import BaselineClassifier
from folktexts.benchmark import Benchmark as ft_benchmark

/Users/mgorecki/opt/miniconda3/envs/monoc-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'folktexts.baseline'

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier

In [5]:
task_name = "ACSIncome"
data_dir = Path.cwd() / "data"
acs_dataset_configs = ft_benchmark.ACS_DATASET_CONFIGS


In [6]:
dataset = ACSDataset.make_from_task(
    task_name, cache_dir=data_dir, **acs_dataset_configs
)

X_train, y_train = dataset.get_train()
X_test, y_test = dataset.get_test()
s_test = None
if dataset.task.sensitive_attribute is not None:
    s_test = dataset.get_sensitive_attribute_data().loc[y_test.index]

Loading ACS data...


In [9]:
base_clf = BaselineClassifier(model_name = 'LogisticRegression', task='ACSIncome', clf_params={'strategy': 'prior', 'penalty': 'l2', 'max_iter': 500})
base_clf

BaselineClassifier(clf_params={'max_iter': 500, 'penalty': 'l2'},
                   model_name='LogisticRegression',
                   task=ACSTaskMetadata(name='ACSIncome',
                                        description='predict whether an '
                                                    "individual's income is "
                                                    'above $50,000',
                                        features=['AGEP', 'COW', 'SCHL', 'MAR',
                                                  'OCCP', 'POBP', 'RELP',
                                                  'WKHP', 'SEX', 'RAC1P'],
                                        target='PINCP',
                                        cols_to_text={'AGEP': <folktexts.col_to_text.ColumnToText o...
                                                                                                 'O',
                                                                                                 'P',
                                                                                                 'Q',
                                                                                                 'R',
                                                                                                 'S',
                                                                                                 'T',
                                                                                                 'U',
                                                                                                 'V',
                                                                                                 'W',
                                                                                                 'X',
                                                                                                 'Y',
                                                                                                 'Z')),
                                        direct_numeric_qa=DirectNumericQA(column='PINCP>50000',
                                                                          text='What '
                                                                               'is '
                                                                               'the '
                                                                               'probability '
                                                                               'that '
                                                                               'this '
                                                                               "person's "
                                                                               'estimated '
                                                                               'yearly '
                                                                               'income '
                                                                               'is '
                                                                               'above '
                                                                               '$50,000 '
                                                                               '?',
                                                                          num_forward_passes=2,
                                                                          answer_probability=True),
                                        _use_numeric_qa=False,
                                        folktables_obj=<folktables.folktables.BasicProblem object at 0x1252053d0>))

In [10]:
base_clf = base_clf.fit(X_train, y_train)

/Users/mgorecki/opt/miniconda3/envs/llm-py311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
base_clf.predict(X_test)

array([0, 0, 0, ..., 0, 1, 0])

In [ ]:

def run_baselines(tasks: list, rerun: bool=False):
    baseline_results_all_tasks = {}
    baseline_risk_scores_all_tasks = {}
    for task_name in tasks:
        print(task_name)
        dataset = ACSDataset.make_from_task(
            task_name, cache_dir=data_dir, **acs_dataset_configs
        )

        X_train, y_train = dataset.get_train()
        X_test, y_test = dataset.get_test()
        s_test = None
        if dataset.task.sensitive_attribute is not None:
            s_test = dataset.get_sensitive_attribute_data().loc[y_test.index]

        print("Run baselines")
        results = {}
        risk_scores = []
        for clf_name, clf in baselines.items():
            clf_path = BASELINE_RESULTS_PATH / f"{clf_name}_task-{task_name}"
            if (clf_path).exists() and not rerun:
                print("Load predictions from file.")
                scores = pd.read_csv(clf_path / f'{task_name}.test_predictions.csv')
                prediction_eval = load_json(path=clf_path / f'{task_name}-results.bench.json')
            else: 
                scores, prediction_eval = fit_and_eval(
                    clf, X_train, y_train, X_test, y_test, s_test, fillna=(clf_name == "LR")
                )
                scores = pd.Series(scores, index=y_test.index, name=f"{clf_name}_task-{task_name}")
                prediction_eval = add_meta_data(prediction_eval, task=task_name, clf_name=clf_name)

                print(f"Save predictions at '{clf_path}'.")
                clf_path.mkdir(parents=True, exist_ok=True)
                scores.to_csv( clf_path / f'{task_name}.test_predictions.csv')
                save_json(obj=prediction_eval, path=clf_path / f'{task_name}-results.bench.json')
                
            results[clf_name] = prediction_eval
            risk_scores.append(scores)

        baseline_results_all_tasks[task_name] = results
        baseline_risk_scores_all_tasks[task_name] = pd.concat(risk_scores, axis=1)

    return baseline_risk_scores_all_tasks, baseline_results_all_tasks